# Porto → Casablanca GPS Remapping
Full pipeline with memory-optimized SparkSession

In [31]:
import sys
print(sys.executable)
import pyspark
print('PySpark version:', pyspark.__version__)

c:\Users\dell\Desktop\big data project\TaaSim-Project\.venv-1\Scripts\python.exe
PySpark version: 3.5.0


## 1. Start SparkSession with enough memory

In [32]:
from pyspark.sql import SparkSession
from pyspark import SparkContext
import os

# Force-clean stale Spark/Py4J state (fixes ConnectionRefusedError after JVM crash)
try:
    if 'spark' in globals() and spark is not None:
        spark.stop()
except Exception:
    pass

SparkSession._instantiatedSession = None
SparkSession._activeSession = None
SparkContext._active_spark_context = None
SparkContext._gateway = None
SparkContext._jvm = None

# Windows Hadoop fix (required when using S3A packages on local Spark)
hadoop_home = os.path.abspath(os.path.join(os.path.expanduser("~"), "hadoop"))
hadoop_home_posix = hadoop_home.replace("\\", "/")
hadoop_bin = os.path.join(hadoop_home, "bin")
os.environ["HADOOP_HOME"] = hadoop_home
os.environ["PATH"] = hadoop_bin + os.pathsep + os.environ.get("PATH", "")
os.environ["SPARK_SUBMIT_OPTS"] = f"-Dhadoop.home.dir={hadoop_home_posix}"
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1' # HADA HOWA LI KAYFIXI TIMEOUT F WINDOWS

# Spark session configured for MinIO (S3A)
spark = SparkSession.builder \
    .appName("PortoCasaRemap") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))
print("Spark version:", spark.version)

HADOOP_HOME: C:\Users\dell\hadoop
Spark version: 3.5.0


## 2. Load the dataset

In [33]:
input_path = "s3a://raw/porto-trips/"
df = spark.read.csv(input_path, header=True, inferSchema=True)
print("Input path:", input_path)
print("Row count:", df.count())
df.printSchema()

Input path: s3a://raw/porto-trips/
Row count: 1710670
root
 |-- TRIP_ID: long (nullable = true)
 |-- CALL_TYPE: string (nullable = true)
 |-- ORIGIN_CALL: integer (nullable = true)
 |-- ORIGIN_STAND: integer (nullable = true)
 |-- TAXI_ID: integer (nullable = true)
 |-- TIMESTAMP: integer (nullable = true)
 |-- DAY_TYPE: string (nullable = true)
 |-- MISSING_DATA: boolean (nullable = true)
 |-- POLYLINE: string (nullable = true)



In [34]:
df.select('TRIP_ID', 'POLYLINE').show(5, truncate=False)

+-------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 3. Parse POLYLINE column into array of coordinates

In [35]:
from pyspark.sql.functions import from_json, col, lit, when, explode
from pyspark.sql.types import ArrayType, DoubleType

polyline_schema = ArrayType(ArrayType(DoubleType()))

df = df.withColumn(
    'coords',
    from_json(col('POLYLINE'), polyline_schema)
)

df.select('TRIP_ID', 'coords').show(3, truncate=False)

+-------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 4. Filter bad rows early (reduces memory pressure)

In [36]:
import csv
import os
import subprocess
import sys

project_root = os.getcwd()
if not os.path.isdir(os.path.join(project_root, 'data')):
    project_root = os.path.abspath(os.path.join(project_root, '..'))

# Keep zone mapping in sync with scripts/generate_zone_mapping.py
generate_script = os.path.join(project_root, 'scripts', 'generate_zone_mapping.py')
subprocess.check_call([sys.executable, generate_script], cwd=project_root)

zone_path = os.path.join(project_root, 'data', 'zone_mapping.csv')
centroids = []
with open(zone_path, 'r', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        centroids.append((float(row['centroid_lon']), float(row['centroid_lat'])))

PORTO_LON_MIN, PORTO_LON_MAX = -8.690, -8.550
PORTO_LAT_MIN, PORTO_LAT_MAX = 41.100, 41.190

CASA_LON_MIN = min(lon for lon, _ in centroids)
CASA_LON_MAX = max(lon for lon, _ in centroids)
CASA_LAT_MIN = min(lat for _, lat in centroids) - 0.002
CASA_LAT_MAX = max(lat for _, lat in centroids) + 0.002

print('Zone mapping regenerated from script:', zone_path)

Zone mapping regenerated from script: c:\Users\dell\Desktop\big data project\TaaSim-Project\data\zone_mapping.csv


## 5. Explode coordinates into separate rows

In [37]:
df = df.withColumn('coord', explode(col('coords'))) \
       .withColumn('porto_lon', col('coord')[0]) \
       .withColumn('porto_lat', col('coord')[1])

df.select('TRIP_ID', 'porto_lon', 'porto_lat').show(5)

+-------------------+---------+---------+
|            TRIP_ID|porto_lon|porto_lat|
+-------------------+---------+---------+
|1372636858620000589|-8.618643|41.141412|
|1372636858620000589|-8.618499|41.141376|
|1372636858620000589|-8.620326| 41.14251|
|1372636858620000589|-8.622153|41.143815|
|1372636858620000589|-8.623953|41.144373|
+-------------------+---------+---------+
only showing top 5 rows



## 6. Define bounding boxes for Porto and Casablanca

In [38]:
porto_lon_min, porto_lon_max = -8.690, -8.550
porto_lat_min, porto_lat_max = 41.100, 41.190
casa_lon_min, casa_lon_max = -7.680, -7.520
casa_lat_min, casa_lat_max = 33.470, 33.600

## 7. Clamp Porto coordinates to bounding box

In [39]:

porto_lon_clamped = when(col('porto_lon') < porto_lon_min, lit(porto_lon_min)) \
    .when(col('porto_lon') > porto_lon_max, lit(porto_lon_max)) \
    .otherwise(col('porto_lon'))

porto_lat_clamped = when(col('porto_lat') < porto_lat_min, lit(porto_lat_min)) \
    .when(col('porto_lat') > porto_lat_max, lit(porto_lat_max)) \
    .otherwise(col('porto_lat'))


## 8. Remap Porto GPS -> Casablanca GPS + OSRM road snapping (test mode)

In [40]:
import requests
from pyspark.sql.functions import collect_list, struct,lit


df_sampled = df.limit(500)

df_sampled = df_sampled.withColumn(
    'raw_casa_lon',
    lit(casa_lon_min) + ((porto_lon_clamped - lit(porto_lon_min)) / lit(porto_lon_max - porto_lon_min)) * lit(casa_lon_max - casa_lon_min)
).withColumn(
    'raw_casa_lat',
    lit(casa_lat_min) + ((porto_lat_clamped - lit(porto_lat_min)) / lit(porto_lat_max - porto_lat_min)) * lit(casa_lat_max - casa_lat_min)
)


trips_raw = df_sampled.groupBy('TRIP_ID') \
    .agg(collect_list(struct('raw_casa_lat', 'raw_casa_lon')).alias('raw_waypoints'))


pdf_trips = trips_raw.toPandas()

def snap_trip_to_road(waypoints):
    snapped = []
    for wp in waypoints:
        lat = wp.raw_casa_lat
        lon = wp.raw_casa_lon
        try:
            url = f"http://127.0.0.1:5000/nearest/v1/driving/{lon},{lat}?number=1"
            r = requests.get(url, timeout=1)
            if r.status_code == 200:
                data = r.json()
                if data.get("code") == "Ok" and data.get("waypoints"):
                    lon = float(data["waypoints"][0]["location"][0])
                    lat = float(data["waypoints"][0]["location"][1])
        except:
            pass
        snapped.append({'casa_lat': float(lat), 'casa_lon': float(lon)})
    return snapped

print("Applying OSRM safely in Pandas...")
print("Formatting data types for Spark...")

pdf_trips['waypoints'] = pdf_trips['raw_waypoints'].apply(snap_trip_to_road)

print("Done! Here is your clean, snapped data f Pandas:")
print(pdf_trips[['TRIP_ID', 'waypoints']].head())

Applying OSRM safely in Pandas...
Formatting data types for Spark...
Done! Here is your clean, snapped data f Pandas:
               TRIP_ID                                          waypoints
0  1372636858620000589  [{'casa_lat': 33.529811, 'casa_lon': -7.598481...
1  1372637303620000596  [{'casa_lat': 33.556323, 'casa_lon': -7.62306}...
2  1372636951620000320  [{'casa_lat': 33.528073, 'casa_lon': -7.592344...
3  1372636854620000520  [{'casa_lat': 33.54508, 'casa_lon': -7.548242}...
4  1372637091620000337  [{'casa_lat': 33.586399, 'casa_lon': -7.629915...


## 9. Group coordinates back by trip (memory-safe — sampled)

In [44]:
import requests
import folium
import random

def get_full_route_from_osrm(waypoints):
    """
    Katsiyfet r-ri7la kamla l OSRM bach yred lina l'geometrie d chanti (Route API)
    """
    if len(waypoints) < 2:
        return []

    # 1. Njm3o n-no9at f string wa7d (Max 50 no9ta bach mayt-plantach OSRM)
    coords_list = [f"{wp.raw_casa_lon},{wp.raw_casa_lat}" for wp in waypoints[:50]]
    coords_str = ";".join(coords_list)
    
    # 2. N-khdmo b ROUTE API f blast Nearest, w ntlbo 'geojson'
    url = f"http://127.0.0.1:5000/route/v1/driving/{coords_str}?geometries=geojson&overview=full"
    
    try:
        r = requests.get(url, timeout=2)
        if r.status_code == 200:
            data = r.json()
            if data.get("code") == "Ok" and data.get("routes"):
                # Hna OSRM kay3tina l'khet l'mzgzez (curves) d chanti kaml
                route_coords = data["routes"][0]["geometry"]["coordinates"]
                # OSRM kayred [Lon, Lat], Folium kaykhedm b [Lat, Lon]
                return [{'casa_lat': p[1], 'casa_lon': p[0]} for p in route_coords]
    except:
        pass
    
    # Ila w93at erreur, nreddo n-no9at l'9dam
    return [{'casa_lat': wp.raw_casa_lat, 'casa_lon': wp.raw_casa_lon} for wp in waypoints]


print("Asking OSRM to draw the full roads... (Wait a few seconds)")
# N-appliqiw l'API jdid
pdf_trips['full_route'] = pdf_trips['raw_waypoints'].apply(get_full_route_from_osrm)

Asking OSRM to draw the full roads... (Wait a few seconds)


In [ ]:
one_trip = pdf_trips["trips_"]
one_trip

TRIP_ID                                        1372636858620000589
raw_waypoints    [(33.52981733333334, -7.598449142857143), (33....
waypoints        [{'casa_lat': 33.529811, 'casa_lon': -7.598481...
full_route       [{'casa_lat': 33.529811, 'casa_lon': -7.598481...
Name: 0, dtype: object

## 10. Interactive map with Folium

In [51]:
import folium

print("Drawing the map with routes, start (Green) and end (Red) points...")
# T-centrer l'khareeta f Casa
m = folium.Map(location=[33.57, -7.60], zoom_start=13)

for _, row in pdf_trips.iterrows():
    # Njbdo r-ri7la l'kamla
    coords = [(p['casa_lat'], p['casa_lon']) for p in row['full_route']]
    if len(coords) < 2:
        continue
    
    # 1. N-rsmou l'Khet d chanti (Zra9)
    folium.PolyLine(
        coords,
        color="#096ACB", 
        weight=2.5,      
        opacity=0.8      
    ).add_to(m)
    
    # 2. N-rsmou n-No9ta dyal L-BEDAYA (Khder)
    start_point = coords[0]
    folium.CircleMarker(
        location=start_point,
        radius=4,            # Kber dyal d-da2ira
        color='green',       # Loun dyal l'khet
        fill=True,
        fill_color='green',  # Loun d l-dakhel
        fill_opacity=0.9,
        popup=f"Start: {row['TRIP_ID']}" # Mlli t-cliqui 3liha t3tik smit r-ri7la
    ).add_to(m)

    # 3. N-rsmou n-No9ta dyal N-NIHAYA (7mer)
    end_point = coords[-1]
    folium.CircleMarker(
        location=end_point,
        radius=4,
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.9,
        popup=f"End: {row['TRIP_ID']}"
    ).add_to(m)

print("Done! L'khareeta wajda, Tmtte3!")
m.save('../data/casa_remap_trips_map.html')

Drawing the map with routes, start (Green) and end (Red) points...
Done! L'khareeta wajda, Tmtte3!


In [55]:
m.save('../data/casa_remap_trips_map.html')

In [57]:
import folium
import requests
from pyspark.sql.functions import col, from_json, explode, lit, when
from pyspark.sql.types import ArrayType, DoubleType

print("Fetching data for TRIP_ID = 1372636858620000589 ...")

# 1. N-jbdou ghir rri7la li bghiti mn l'DataFrame l'asli (df)
trip_id_to_test = 1372637091620000337
df_single = df.filter(col('TRIP_ID') == trip_id_to_test)

# 2. N-saybo l'I7datiyat l'kham (Math interpolation)
if 'coords' not in df_single.columns:
    polyline_schema = ArrayType(ArrayType(DoubleType()))
    df_single = df_single.withColumn('coords', from_json(col('POLYLINE'), polyline_schema))

df_single = df_single.withColumn('coord', explode(col('coords'))) \
       .withColumn('porto_lon', col('coord')[0]) \
       .withColumn('porto_lat', col('coord')[1])

porto_lon_min, porto_lon_max = -8.690, -8.550
porto_lat_min, porto_lat_max = 41.100, 41.190
casa_lon_min, casa_lon_max = -7.680, -7.520
casa_lat_min, casa_lat_max = 33.470, 33.600

porto_lon_clamped = when(col('porto_lon') < porto_lon_min, lit(porto_lon_min)) \
    .when(col('porto_lon') > porto_lon_max, lit(porto_lon_max)) \
    .otherwise(col('porto_lon'))
porto_lat_clamped = when(col('porto_lat') < porto_lat_min, lit(porto_lat_min)) \
    .when(col('porto_lat') > porto_lat_max, lit(porto_lat_max)) \
    .otherwise(col('porto_lat'))

df_single = df_single.withColumn(
    'raw_casa_lon',
    lit(casa_lon_min) + ((porto_lon_clamped - lit(porto_lon_min)) / lit(porto_lon_max - porto_lon_min)) * lit(casa_lon_max - casa_lon_min)
).withColumn(
    'raw_casa_lat',
    lit(casa_lat_min) + ((porto_lat_clamped - lit(porto_lat_min)) / lit(porto_lat_max - porto_lat_min)) * lit(casa_lat_max - casa_lat_min)
)

# 3. N-redouha Pandas bach n-rsmoha w nsowlo OSRM
pdf_single = df_single.select('raw_casa_lat', 'raw_casa_lon').toPandas()

# N-jm3o n-no9at lkham f liste
raw_coords = [(row['raw_casa_lat'], row['raw_casa_lon']) for _, row in pdf_single.iterrows()]

# 4. N-sowlo OSRM 3la chanti d بصح (Map Matching)
osrm_coords = []
if len(raw_coords) >= 2:
    print("Calculating Map Matching with OSRM...")
    # Nsifto 50 no9ta l OSRM
    coords_str = ";".join([f"{lon},{lat}" for lat, lon in raw_coords[:50]]) 
    url = f"http://127.0.0.1:5000/route/v1/driving/{coords_str}?geometries=geojson&overview=full"
    try:
        r = requests.get(url, timeout=2)
        if r.status_code == 200:
            data = r.json()
            if data.get("code") == "Ok" and data.get("routes"):
                # N-jbdo l'geometrie d chanti kaml
                route_geom = data["routes"][0]["geometry"]["coordinates"]
                osrm_coords = [(p[1], p[0]) for p in route_geom] # Folium kaykhedm b [Lat, Lon]
    except Exception as e:
        print("OSRM Error:", e)

# 5. ==========================================
#    N-RSMO L'KHAREETA D L'MOQARANA
# ==========================================
print("Drawing Comparison Map...")
m = folium.Map(location=[raw_coords[0][0], raw_coords[0][1]], zoom_start=16)

# A) Rsem l'Khet l'Kham (Bla OSRM) - Loun 7mer mt9te3
folium.PolyLine(
    raw_coords,
    color='#FF3333', 
    weight=3,
    opacity=0.7,
    dash_array='10, 10', # Khet mt9te3 bach yban blli machi 79i9i
    tooltip="Raw Data (Before OSRM) - Passes through buildings!"
).add_to(m)

# B) Rsem l'Khet l'M9add (B OSRM) - Loun Khder Neon
if osrm_coords:
    folium.PolyLine(
        osrm_coords,
        color='#00FF00', 
        weight=5,
        opacity=0.9,
        tooltip="Mapped Data (After OSRM) - Follows the road!"
    ).add_to(m)

# C) 3alamat dyal l'Bedaya w N-Nihaya
folium.CircleMarker(location=raw_coords[0], radius=5, color='white', fill_color='yellow', fill_opacity=1, popup="Start").add_to(m)
folium.CircleMarker(location=raw_coords[-1], radius=5, color='white', fill_color='white', fill_opacity=1, popup="End").add_to(m)

print("Done! Tmtte3 b l'far9!")
m.save('../data/casa_remap_trips_map.html')

Fetching data for TRIP_ID = 1372636858620000589 ...
Calculating Map Matching with OSRM...
Drawing Comparison Map...
Done! Tmtte3 b l'far9!
